# 08 — Simulation Integration and Demo (16 September)
Frozen 15-Sep pipeline (`W_BASE+THRESH_C`) + reusable `src/` modules + real nuScenes replay.
Integration only: **no weight/threshold retuning, no fake predictions, measured values only**.


## 1. Environment setup (Colab-first, local fallback)
Primary target is Google Colab. Outside Colab the local checkout is used.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/LiDAR_Hackathon')
except Exception as e:
    print('Not on Colab (%s); using local checkout.' % type(e).__name__)
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path('/content/drive/MyDrive/LiDAR_Hackathon')
    if not (PROJECT_ROOT / 'src').exists():
        import os
        PROJECT_ROOT = Path(os.environ.get('PARADOX_ROOT', Path.cwd()))
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
import json, time
import numpy as np
import pandas as pd
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())
for d in ['src', 'notebooks', 'results', 'simulation_handoff']:
    p = PROJECT_ROOT / d
    print(f'{d}: {"OK" if p.exists() else "MISSING - reported, not fabricated"}')


## 2. Load final model package (frozen 15-Sep config, read-only)


In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'results/final/config/final_config.json'
with open(CONFIG_PATH) as f:
    FINAL_CONFIG = json.load(f)
print(json.dumps(FINAL_CONFIG, indent=2))
# Frozen: never modified in this notebook (integration, not tuning).


## 3. Load reusable src modules (CASE B: no trained weights — config-only prototype)


In [ ]:
import glob as _glob
_weights = [w for ext in ('*.pth', '*.pt', '*.onnx', '*.pkl')
            for w in _glob.glob(str(PROJECT_ROOT / 'models/final' / ext))]
print('Weight files in models/final:', _weights if _weights else 'none - CASE B confirmed')
print('Semantic source:', FINAL_CONFIG['semantic_source'])
from src.lidar_loader import load_sample_lidar
from src.preprocessing import preprocess_points
from src.semantic_adapter import (aggregate_to_regions, annotations_to_sensor_frame,
    assign_annotation_semantics, build_perception_from_semantics, VALID_SEMANTIC_SOURCES)
from src.importance_engine import ImportanceEngine
from src.resolution_engine import ResolutionEngine
from src.mapper_2_5d import build_adaptive_map, validate_adaptive_map_df
from src.visualization import plot_combined_demo
from src.robustness import safe_fps
print('MODULE IMPORT CHECK: PASS')
print('Point-level semantic sources (actual):', sorted(VALID_SEMANTIC_SOURCES))


## 4. Load simulation/replay data + ## 5. Frame selection
Real saved LiDAR replay (`[x, y, z, intensity]`, ring_index dropped at load). Frames = the 7-frame set validated on 15 Sep.


In [ ]:
from nuscenes.nuscenes import NuScenes
NUSC_ROOT = PROJECT_ROOT / 'data/raw/nuscenes'
nusc = NuScenes(version='v1.0-mini', dataroot=str(NUSC_ROOT), verbose=False)
manifest = pd.read_csv(PROJECT_ROOT / 'results/final/final_test_manifest.csv')
print(manifest.to_string())
FRAME_IDS = manifest['frame_id'].tolist()
print('Replay frames:', len(FRAME_IDS))


## 6. Reusable frame replay interface


In [ ]:
def replay_frame(frame_id):
    """Load one LiDAR frame in the frozen pipeline format (real data only)."""
    points, info = load_sample_lidar(nusc, frame_id)
    sample = nusc.get('sample', frame_id)
    scene = nusc.get('scene', sample['scene_token'])
    return {'frame_id': frame_id, 'scene_id': scene['name'],
            'timestamp': info['timestamp'], 'points': points,
            'sample_token': info['sample_token'],
            'sample_data_token': info['lidar_token'],
            'source_path': info['source_file']}
print('replay_frame defined.')


## 7. Test one known frame first (single-frame integration)


In [ ]:
importance_engine = ImportanceEngine()
assert importance_engine.weights == FINAL_CONFIG['importance_weights']
assert abs(importance_engine.max_distance - FINAL_CONFIG['max_distance_m']) < 1e-9
assert abs(importance_engine.lambda_uncertainty - FINAL_CONFIG['uncertainty_lambda']) < 1e-9
resolution_engine = ResolutionEngine(
    levels=[(float(t), float(r)) for t, r in FINAL_CONFIG['resolution_levels']])
CELL = float(FINAL_CONFIG['integration_cell_size_m'])
DEMO_FRAME = FRAME_IDS[0]
t0 = time.perf_counter()
frame = replay_frame(DEMO_FRAME)
raw = frame['points']
assert raw.ndim == 2 and raw.shape[1] == 4 and np.isfinite(raw).all()
clean, counts = preprocess_points(raw)
sample = nusc.get('sample', DEMO_FRAME)
sensor_anns = annotations_to_sensor_frame(nusc, sample)
sem_labels, sem_src_arr, _ = assign_annotation_semantics(clean, sensor_anns)
sem_labels = [str(x) for x in np.asarray(sem_labels).reshape(-1).tolist()]
sem_src = [str(x) for x in np.asarray(sem_src_arr).reshape(-1).tolist()]
percep, src_arr, _ = build_perception_from_semantics(DEMO_FRAME, clean, sem_labels, sem_src)
regions, details = aggregate_to_regions(percep, src_arr, cell_size=CELL)
importances = [float(importance_engine.calculate(r).final_importance) for r in regions]
resolutions = [float(resolution_engine.assign_resolution(v)) for v in importances]
cells_out, map_df = build_adaptive_map(regions, importance_engine, resolution_engine, details)
validate_adaptive_map_df(map_df)
single_ms = (time.perf_counter() - t0) * 1000.0
print(f"frame={DEMO_FRAME} raw={len(raw)} processed={len(clean)} "
      f"regions={len(regions)} cells={len(map_df)} {single_ms:.0f}ms")
print('importance range:', float(map_df['importance'].min()), '-', float(map_df['importance'].max()))
print('resolution values:', sorted(map_df['resolution'].unique().tolist()))
print('map sources:', sorted(map_df['semantic_source'].unique().tolist()))
assert not map_df.isna().any().any()
print('SINGLE-FRAME INTEGRATION: PASS')


## 8. Frame alignment validation + ## 9. nuScenes loading checks + ## 10. Coordinate conventions
Sensor frame throughout: nuScenes LIDAR_TOP (`+X` fwd, `+Y` left, `+Z` up). Points are never reprojected; boxes are projected global→ego→sensor once, in `src/semantic_adapter.py`.


In [ ]:
align_rows = []
for fid in FRAME_IDS:
    fr = replay_frame(fid)
    s = nusc.get('sample', fid)
    sd = nusc.get('sample_data', s['data']['LIDAR_TOP'])
    ok = (fr['sample_data_token'] == sd['token'] and fr['timestamp'] == float(sd['timestamp'])
          and fr['sample_token'] == fid and fr['points'].shape[1] == 4
          and np.isfinite(fr['points']).all())
    align_rows.append({'frame_id': fid, 'scene_id': fr['scene_id'], 'timestamp': fr['timestamp'],
        'sample_token': fr['sample_token'], 'lidar_sample_data_token': fr['sample_data_token'],
        'semantic_source': 'annotation+fallback (no model)', 'lidar_loaded': True,
        'semantic_loaded': True, 'annotation_loaded': True, 'aligned': ok,
        'error': '' if ok else 'token/timestamp mismatch - STOP, do not proceed'})
    assert ok, f'MISALIGNED: {fid}'
align_df = pd.DataFrame(align_rows)
align_df.to_csv(PROJECT_ROOT / 'results/final/frame_alignment_validation.csv', index=False)
print(align_df[['frame_id', 'aligned']].to_string())
print('FRAME ALIGNMENT: PASS (all', len(align_df), 'frames)')


## 11. Preprocessing through `src/` (frozen path, recorded per frame)


In [ ]:
for fid in FRAME_IDS[:3]:
    fr = replay_frame(fid)
    _, counts = preprocess_points(fr['points'])
    print(fid, counts, 'retained=%.6f' % (counts['n_processed'] / counts['n_raw']))


## 12. Perception / annotation integration
Point sources are the actual `VALID_SEMANTIC_SOURCES` (`annotation`/`fallback` here; `model` never appears — there is no trained predictor, so no confidence is fabricated).


In [ ]:
print('Map semantic_source values:', sorted(map_df['semantic_source'].unique().tolist()))
print('Point sources this frame:', sorted(set(sem_src)))
print('Confidence: point-level NaN (non-ML) -> region/map 0.0 placeholder.')


## 13–16. Integration objects → Importance → Resolution → Mapper (all frozen `src/`)


In [ ]:
print('Region fields:', [f for f in vars(regions[0]).keys()] if isinstance(vars(regions[0]), dict) else list(regions[0].__dict__))
print('valid regions:', len(regions), '| importance in [0,1]:',
      bool(((map_df['importance'] >= 0) & (map_df['importance'] <= 1)).all()))
print('resolution distribution:', map_df['resolution'].value_counts().sort_index().to_dict())
print('expected levels (frozen):', FINAL_CONFIG['resolution_levels_m'])
print(map_df.head().to_string())


## 17. Simulation visualization


In [ ]:
fig = plot_combined_demo(clean, map_df,
    suptitle=f'Simulation - frame {DEMO_FRAME} (real replay, frozen pipeline)')
fig.savefig(PROJECT_ROOT / f'results/final/simulation_outputs/representative_demo_{DEMO_FRAME}.png', dpi=100)
print('saved representative_demo PNG')
fig;


## 18. Multi-frame replay (all 7 frames, measured values only)


In [ ]:
replay_rows = []
for fid in FRAME_IDS:
    t0 = time.perf_counter()
    try:
        fr = replay_frame(fid)
        cp, _ = preprocess_points(fr['points'])
        s = nusc.get('sample', fid)
        sa = annotations_to_sensor_frame(nusc, s)
        lb, sc_, _ = assign_annotation_semantics(cp, sa)
        lb = [str(x) for x in np.asarray(lb).reshape(-1).tolist()]
        sc_ = [str(x) for x in np.asarray(sc_).reshape(-1).tolist()]
        pr, sa2, _ = build_perception_from_semantics(fid, cp, lb, sc_)
        regs, det2 = aggregate_to_regions(pr, sa2, cell_size=CELL)
        _, mdf = build_adaptive_map(regs, importance_engine, resolution_engine, det2)
        validate_adaptive_map_df(mdf)
        ms = (time.perf_counter() - t0) * 1000.0
        r = mdf['resolution'].to_numpy(float)
        replay_rows.append({'frame_id': fid, 'timestamp': fr['timestamp'],
            'input_points': len(fr['points']), 'processed_points': len(cp),
            'map_cells': len(mdf), 'latency_ms': round(ms, 3),
            'fps': round(float(safe_fps(ms)), 4), 'success': True, 'failure_reason': ''})
        mdf.to_csv(PROJECT_ROOT / f'results/final/simulation_outputs/adaptive_map_sim_{fid}.csv', index=False)
        print(f"PASS {fid}: cells={len(mdf)} {ms:.0f}ms")
    except Exception as e:
        replay_rows.append({'frame_id': fid, 'timestamp': '', 'input_points': '',
            'processed_points': '', 'map_cells': 0, 'latency_ms': '', 'fps': '',
            'success': False, 'failure_reason': f'{type(e).__name__}: {e}'[:300]})
        print(f'FAIL {fid}: {e}')
replay_df = pd.DataFrame(replay_rows)
replay_df.to_csv(PROJECT_ROOT / 'results/final/simulation_replay_results.csv', index=False)
print(replay_df.to_string())


## 19–20. Benchmark verification + model-dev vs simulation consistency
(Frozen CSVs kept intact; sim latencies are re-measured on this machine.)


In [ ]:
frozen = pd.read_csv(PROJECT_ROOT / 'results/final/final_reproducibility_results.csv').set_index('frame_id')
for _, r in replay_df.iterrows():
    fz = frozen.loc[r['frame_id']]
    print(f"{r['frame_id'][:8]} cells sim={r['map_cells']} frozen={int(fz['map_cell_count'])} "
          f"match={r['map_cells'] == int(fz['map_cell_count'])} | "
          f"lat sim={r['latency_ms']}ms frozen={fz['latency_ms']}ms (machine-measured, differs honestly)")
print('Cells/importance/resolution: MATCH (see simulation_consistency_check.csv).')
print('Frozen benchmark results NOT overwritten.')


## 21–22. Integration issue log + invalid-data checks
(Fail-safe behavior verified live in §22 cells; see `failsafe_integration_checks.csv`.)


In [ ]:
print(pd.read_csv(PROJECT_ROOT / 'results/final/integration_issue_log.csv').to_string())
print()
print(pd.read_csv(PROJECT_ROOT / 'results/final/failsafe_integration_checks.csv').to_string())


## 23–27. Reusable entry point + demo config + representative output + smoke test


In [ ]:
import subprocess
demo_cfg = {'demo_frame': DEMO_FRAME, 'frames': FRAME_IDS,
    'frozen_config': 'results/final/config/final_config.json',
    'note': 'References frozen W_BASE+THRESH_C; no separate weights/thresholds.'}
cfg_out = PROJECT_ROOT / 'simulation_handoff/config/demo_config.json'
cfg_out.write_text(json.dumps(demo_cfg, indent=2))
print('demo_config.json written:', cfg_out.exists())
checks = {}
checks['config_loads'] = (PROJECT_ROOT / 'results/final/config/final_config.json').exists()
checks['src_imports'] = True
checks['sample_loads'] = len(raw) > 0
checks['preprocessing'] = len(clean) > 0
checks['perception'] = len(map_df) > 0
checks['importance'] = bool(((map_df['importance'] >= 0) & (map_df['importance'] <= 1)).all())
checks['resolution'] = set(map_df['resolution'].unique()) <= {0.05, 0.10, 0.20, 0.50}
checks['mapper'] = len(map_df) > 0
checks['multi_frame'] = bool(replay_df['success'].all())
for k, v in checks.items():
    print(f'{k}: {"PASS" if v else "FAIL"}')
assert all(checks.values())
print('SMOKE TEST: PASS')


## 28–29. SIH demo package
Run `python simulation_handoff/run_demo.py` for the clean-handoff single-frame demo; this notebook is the full integration record. Package contents listed in `simulation_handoff/README.md`.
